# PRFT
Product Right First Time. The products with a released / pass status at the time of initial process (Without rework). 100% running properly as well from material supply to end product / goods.


In [22]:
import pandas as pd
import numpy as np

monthDict = {'January':1,'February':2,'March':3, 'April':4, 'May':5, 'June':6, 'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12}

df=pd.read_excel("../Dataset/prod_output.xlsx",header=0)
dfA=pd.DataFrame(df)
dfB=dfA.reset_index()
dfB['Opr_daily'] = dfB['Opr_daily'].fillna(0)
dfB['ProcessDate'] = (pd.to_datetime(dfB['ProcessDate'])).dt.strftime('%d-%m-%Y')
dfB['month']=(pd.to_datetime(dfB['ProcessDate'])).dt.strftime('%B')
section=["COLORING","TUBING","STRANDING","JACKETING","ARMORING"]

# Standarizing the data.
prodStat=["released","not-released","rework","direct-pass"]
# Set rework and direct-pass as a not-released product
dfC=dfB[~dfB['RepoStatus'].isin(['Mc-off', 'Cancel', 'Hold'])]
directPassToNotReleased=dfB[dfB['RepoStatus']=="Direct-pass"]['RepoStatus'].unique()
dfC['RepoStatus']=dfC.apply(lambda row:'Not-released' if row['RepoStatus'] in directPassToNotReleased else row['RepoStatus'],axis=1)
reworkToNotreleased=dfC[dfC['RepoStatus']=="Rework"]['RepoStatus'].unique()
dfC['RepoStatus']=dfC.apply(lambda row:'Not-released' if row['RepoStatus'] in reworkToNotreleased else row['RepoStatus'],axis=1)
pendingToNotreleased=dfC[dfC['RepoStatus']=="Pending"]['RepoStatus'].unique()
dfC['RepoStatus']=dfC.apply(lambda row:'Not-released' if row['RepoStatus'] in pendingToNotreleased else row['RepoStatus'],axis=1)

prftA = pd.MultiIndex.from_product([section, monthDict], names=['GroupOrMachine', 'month'])
prftB = pd.DataFrame(index=prftA).reset_index()
prftC = prftB.merge(dfC, on=['GroupOrMachine', 'month'], how='left', indicator=True)
prftC['total'] = (prftC['_merge'] == 'both').astype(int)
prftC.drop(columns=['_merge'], inplace=True)
prftD = (prftC.groupby(['GroupOrMachine', 'month'])[['Qty_CTN','Qty_NG_kgs']]
    .sum()
    .reset_index()
    .rename(columns={
        "Qty_CTN":"Total Good product",
        "Qty_Ng_kgs":"Total NG product"
    })
)
print(prftD)

C:\Users\Ronniawan Fadhlillah\AppData\Local\Temp\ipykernel_15880\45840571.py:11: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dfB['month']=(pd.to_datetime(dfB['ProcessDate'])).dt.strftime('%B')


   GroupOrMachine      month  Total Good product  Qty_NG_kgs
0        ARMORING      April              566.00      111.50
1        ARMORING     August              826.00      107.20
2        ARMORING   December                0.00        0.00
3        ARMORING   February              624.00       60.20
4        ARMORING    January              616.00       82.40
5        ARMORING       July             1239.00      122.20
6        ARMORING       June              722.00       85.40
7        ARMORING      March              513.00      178.76
8        ARMORING        May             1559.00      259.60
9        ARMORING   November                0.00        0.00
10       ARMORING    October              643.00      111.90
11       ARMORING  September             1305.00      197.70
12       COLORING      April                0.00        0.00
13       COLORING     August              114.78       34.00
14       COLORING   December                0.00        0.00
15       COLORING   Febr